# Creation of FAIRML metadata

This step implements **FAIR4ML-compliant** metadata generation for the trained machine learning models. 

The objective is to ensure that each model is fully documented, reproducible, and reusable according to FAIR principles. 


A structured metadata template is automatically generated directly from the trained scikit-learn model object and exported in machine-readable JSON format.


The metadata includes:
- algorithm name
- software library and version
- model hyperparameters
- evaluation metrics
- references to the training dataset (DOI)


Information such as the intended use of the model and known limitations are provided.

To improve reproducibility and reduce manual documentation errors, 
model hyperparameters are extracted automatically using the `get_params()` method provided by scikit-learn. 

The generated FAIR4ML metadata file is stored alongside the trained model artifact
and are referenced within the RO-Crate and TUWRD deposit workflow.



In [2]:
import sys
import os

import json
import sklearn
from datetime import datetime
from pathlib import Path
import pickle

sys.path.append(os.path.abspath(".."))

from src.preprocessing import *
from src.models import *
from src.evaluation import *
from src.utils import *

from sklearn.model_selection import train_test_split
import pandas as pd

### Loading saved models

In [39]:
with open('../outputs/models/model_multi_randomforest.pkl','rb') as f:
    model_multi = pickle.load(f)

with open('../outputs/models/model_cd_randomforest.pkl','rb') as f:
    model_cd = pickle.load(f)

params = model_cd.get_params()
print(params)

### Define Metadata function builder FAIRML compliant

In [33]:
'''
model_data = {
    'model_name' :  '',
    'intended_use' : '',
    'limitation' : '',
    'metrics' : 'mae,rms',
    'mltask' : 'regression'
}

dataset_data = { 'dataset_name' : '' ,
                 'doi' : '',
                 'publication' : '',
                 'author' : ''
}
'''
    
def generate_fair4ml_metadata(
    model,
    model_data,
    dataset_data,
    output_file=None, ):
    
    """
    Generate FAIR4ML metadata in JSON format 
    from a trained ML model.
    """

    metadata = {
        "fair4ml:name" : model_data['model_name'],
        "fair4ml:dateCreated": datetime.utcnow().isoformat() + "Z",
        "fair4ml:modelDetails": {
            "name": type(model).__name__,
            "library": "scikit-learn",
            "library_version": sklearn.__version__,
        },

        "fair4ml:trainingData": {
            #"name": dataset_data['name'],
            #"author": dataset_data['author'],
            "doi": dataset_data['doi'],
            #"publication": dataset_data['publication'],
            
        },

        #"hyperparameters": model.get_params(),
        "fair4ml:performance": evaluation_metrics,
        "fair4ml:intendedUse": model_data['intended_use'],
        "fair4ml:mlTask": model_data['mltask'],
        "fair4ml:limitation": model_data['limitation'],
    }

    # Save JSON file
    if output_file is not None:
        output_path = Path(output_file)

        with open(output_path, "w") as f:
            json.dump(metadata, f, indent=4)

        print(f"FAIR4ML metadata saved to: {output_path}")

    return metadata

In [ ]:
### Create FAIRML compliant JSON files for the models 

In [42]:
# Model 1: Cd Prediction
model = model_cd
model_name = 'sklearn_randomForest'
dataset_name = 'Concentrations of major ions in wet precipitation samples in Austria'
doi = '10.48436/b0g4h-rv840.' 
evaluation_metrics = 'rms,mae'
mltask = 'Regression'
intended_use = 'Prediction of heavy metals in precipitations'
known_limitations = "Highly inaccurate, check with in-situ measurements whenever possible" + "Sparse spatial coverage." + "Heavy metal measurements contain missing values." + "No validation outside the training period." 
author = 'at Al.'
publication = 'Journal'


model_data = {
    'model_name' :  '',
    'intended_use' : intended_use,
    'limitation' : known_limitations,
    'metrics' : evaluation_metrics,
    'mltask' : mltask
}

dataset_data = { 'dataset_name' : dataset_name ,
                 'doi' : doi,
                 'publication' : publication,
                 'author' : author
}


generate_FAIRML = generate_fair4ml_metadata(
    model,
    model_data,
    dataset_data,
    output_file='../outputs/metadata/FAIRML_model_cd'
)


# Model 2: MultiPrediction
model = model_multi
generate_FAIRML = generate_fair4ml_metadata(
    model,
    model_data,
    dataset_data,
    output_file='../outputs/metadata/FAIRML_model_multi'
)



FAIR4ML metadata saved to: ../outputs/metadata/FAIRML_model_cd
FAIR4ML metadata saved to: ../outputs/metadata/FAIRML_model_multi
